In [49]:
# !pip install pyspark
# !pip install "pandas<3.0"

In [50]:
import os
import subprocess
import psycopg2
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from pyspark.sql import SparkSession
from config.database import DB_CONFIG
from pyspark.sql.functions import sum, count, first
from pyspark.sql.functions import col, avg, first, when, lit
from pyspark.sql.functions import (
    col,
    year,
    month,
    dayofmonth,
    quarter,
    weekofyear,
    dayofweek,
    to_date,
    date_format,
    when
)


In [51]:
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = r"C:\Program Files\Java\jdk-17\bin;" + os.environ["PATH"]

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode())

JAVA_HOME: C:\Program Files\Java\jdk-17
java version "17.0.12" 2024-07-16 LTS
Java(TM) SE Runtime Environment (build 17.0.12+8-LTS-286)
Java HotSpot(TM) 64-Bit Server VM (build 17.0.12+8-LTS-286, mixed mode, sharing)



In [52]:


jar_path = r"C:\Users\YapJack\AppData\Roaming\DBeaverData\drivers\maven\maven-central\org.postgresql\postgresql-42.7.11.jar"

print(Path(jar_path).exists())   # Should print True

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("silver_to_gold")
    .config("spark.jars", jar_path)
    .getOrCreate()
)

True


In [53]:
jdbc_url = (
    f"jdbc:postgresql://"
    f"{DB_CONFIG['host']}:{DB_CONFIG['port']}/"
    f"{DB_CONFIG['database']}"
)

In [54]:
connection_properties = {
    "user": DB_CONFIG["user"],
    "password": DB_CONFIG["password"],
    "driver": "org.postgresql.Driver"
}

In [55]:
tables = [
    "customers",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers",
    "geolocation"
]

dfs = {}

for table in tables:
    dfs[table] = (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", f"silver.{table}")
        .option("user", DB_CONFIG["user"])
        .option("password", DB_CONFIG["password"])
        .option("driver", "org.postgresql.Driver")
        .load()
    )

# print(dfs["customers"].count())
# dfs["customers"].show()

## Dim_Customers Table

In [56]:
customer_df = dfs["customers"]
geolocation_df = dfs["geolocation"]

dim_customer = (
    customer_df.alias("c")
    .join(
        geolocation_df.alias("g"),
        customer_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,
        "left"
        )
    .select(
        col("c.customer_id").alias("customer_id"),
        col("c.customer_unique_id").alias("customer_unique_id"),
        col("c.customer_zip_code_prefix").alias("customer_zip_code_prefix"),
        col("g.geolocation_lat").alias("customer_lat"),
        col("g.geolocation_lng").alias("customer_lng"),
        col("g.geolocation_city").alias("customer_city"),
        col("g.geolocation_state").alias("customer_state")
        )
    )

# dim_customer.show()

In [57]:
dim_customer.filter(dim_customer["customer_lng"].isNull()).show()

+--------------------+--------------------+------------------------+------------+------------+-------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_lat|customer_lng|customer_city|customer_state|
+--------------------+--------------------+------------------------+------------+------------+-------------+--------------+
|969464fb93c7e359f...|a0b900ce3e523b7de...|                   73091|        NULL|        NULL|         NULL|          NULL|
|dfe533d0707629488...|1ac4011bec2c0d9ce...|                   71995|        NULL|        NULL|         NULL|          NULL|
|1a2d9dfdc193429a1...|1ac4011bec2c0d9ce...|                   71995|        NULL|        NULL|         NULL|          NULL|
|c67f83708340f8841...|7155c2dc2de010b1e...|                   71995|        NULL|        NULL|         NULL|          NULL|
|219f488f72dad40f8...|016e68615e12298a0...|                   59547|        NULL|        NULL|         NULL|          NULL|
|e37623a

## Dim_Sellers Table

In [58]:
seller_df = dfs["sellers"]
geolocation_df = dfs["geolocation"]

dim_seller = (
    seller_df.alias("s")
    .join(
        geolocation_df.alias("g"),
        seller_df.seller_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,
        "left"
        )
    .select(
        col("s.seller_id").alias("seller_id"),
        col("s.seller_zip_code_prefix").alias("seller_zip_code_prefix"),
        col("g.geolocation_lat").alias("seller_lat"),
        col("g.geolocation_lng").alias("seller_lng"),
        col("g.geolocation_city").alias("seller_city"),
        col("g.geolocation_state").alias("seller_state")
        )
    )

# dim_seller.show()

In [59]:
# from pyspark.sql.functions import col

# dim_seller.filter(
#     col("seller_id").isNull() |
#     col("seller_zip_code_prefix").isNull() |
#     col("seller_city").isNull() |
#     col("seller_state").isNull()
# ).show(truncate=False)

## Dim_Products Table

In [60]:
products_df = dfs["products"]
dim_product = products_df

## Dim_date Table

In [61]:
order_items_df = dfs["order_items"]
orders_df = dfs["orders"]

dates = (
    order_items_df.select(col("shipping_limit_date").alias("date"))
    .union(
        orders_df.select(col("order_purchase_timestamp").alias("date"))
    )
    .union(
        orders_df.select(col("order_approved_at").alias("date"))
    )
    .union(
        orders_df.select(col("order_delivered_carrier_date").alias("date"))
    )
    .union(
        orders_df.select(col("order_delivered_customer_date").alias("date"))
    )
   .union(
        orders_df.select(col("order_estimated_delivery_date").alias("date"))
    )
    .dropna()
    .distinct()
)


dim_date = (
    dates
    .withColumn("full_date", to_date(col("date")))
    .drop("date")
    .dropDuplicates(["full_date"])
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("day_of_week", date_format(col("full_date"), "EEEE"))
    .withColumn("day_of_month", dayofmonth(col("full_date")))
    .withColumn("week_of_year", weekofyear(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("year", year(col("full_date")))
    .withColumn(
        "is_weekend",
        when(dayofweek(col("full_date")).isin(1, 7), True).otherwise(False)
    )
    .select(
        "date_key",
        "full_date",
        "day_of_week",
        "day_of_month",
        "week_of_year",
        "month",
        "quarter",
        "year",
        "is_weekend"
    )
)

## Fact Table Join

In [62]:
order_items_df = dfs["order_items"]
# order_items_df
orders_df = dfs["orders"]
#orders_df
# orders_df

In [63]:
grouped_order_items_df = (
    order_items_df
    .groupBy("order_id", "product_id")
    .agg(
        count("*").alias("unit_ordered"),
        first("seller_id").alias("seller_id"),
        first("shipping_limit_date").alias("shipping_limit_date"),
        sum("price").alias("total_price"),
        sum("freight_value").alias("total_freight_value"),
        
    )
)


grouped_order_items_df = (
    grouped_order_items_df
    .withColumn(
        "revenue",
        col("total_price") + col("total_freight_value")
    )
)
# grouped_order_items_df.show()

The fact table is grouped by order_id and product_id. Since price and freight_value in the Silver layer are recorded at the individual order-item level, aggregating by these columns preserves the correct totals by summing the values for each (order_id, product_id) combination. Therefore, the grouping does not affect the accuracy of the price and freight calculations.

In [64]:
fact_order_sales = (grouped_order_items_df.alias("goi")
                    .join(
                        orders_df.alias("o"),
                        col("o.order_id") == col("goi.order_id"),
                        "left"
                    )
                    .select(
                    col("goi.order_id").alias("order_id"),
                    col("goi.product_id").alias("product_id"),
                    col("goi.seller_id").alias("seller_id"),
                    col("o.customer_id").alias("customer_id"),
                    col("goi.unit_ordered").alias("unit_ordered"),
                    col("goi.total_price").alias("total_price"),
                    col("goi.total_freight_value").alias("total_freight_value"),
                    col("goi.revenue").alias("revenue"),
                    col("goi.shipping_limit_date").alias("shipping_limit_date"), 
                    col("o.order_purchase_timestamp").alias("order_purchase_timestamp"),  
                    col("o.order_approved_at").alias("order_approved_at"),  
                    col("o.order_delivered_carrier_date").alias("order_delivered_carrier_date"), 
                    col("o.order_delivered_customer_date").alias("order_delivered_customer_date"), 
                    col("o.order_estimated_delivery_date").alias("order_estimated_delivery_date"),  
                    )
                    
                )

In [65]:
# Role-playing date dimensions
shipping = dim_date.alias("shipping")
purchase_date = dim_date.alias("purchase")
approved_date = dim_date.alias("approved")
carrier_date = dim_date.alias("carrier")
delivered_date = dim_date.alias("delivered")
estimated_date = dim_date.alias("estimated")

fact_order_sales = (
    grouped_order_items_df.alias("goi")

    # Join Orders
    .join(
        orders_df.alias("o"),
        col("goi.order_id") == col("o.order_id"),
        "left"
    )

    # Shipping Date
    .join(
        shipping,
        to_date(col("goi.shipping_limit_date")) == col("shipping.full_date"),
        "left"
    )

    # Purchase Date
    .join(
        purchase_date,
        to_date(col("o.order_purchase_timestamp")) == col("purchase.full_date"),
        "left"
    )

    # Approved Date
    .join(
        approved_date,
        to_date(col("o.order_approved_at")) == col("approved.full_date"),
        "left"
    )

    # Carrier Date
    .join(
        carrier_date,
        to_date(col("o.order_delivered_carrier_date")) == col("carrier.full_date"),
        "left"
    )

    # Delivered Date
    .join(
        delivered_date,
        to_date(col("o.order_delivered_customer_date")) == col("delivered.full_date"),
        "left"
    )

    # Estimated Delivery Date
    .join(
        estimated_date,
        to_date(col("o.order_estimated_delivery_date")) == col("estimated.full_date"),
        "left"
    )

    .select(
        col("goi.order_id").alias("order_id"),
        col("goi.product_id").alias("product_id"),
        col("goi.seller_id").alias("seller_id"),
        col("o.customer_id").alias("customer_id"),

        col("purchase.date_key").alias("purchase_date_key"),
        col("approved.date_key").alias("approved_date_key"),
        col("carrier.date_key").alias("carrier_date_key"),
        col("delivered.date_key").alias("delivered_date_key"),
        col("estimated.date_key").alias("estimated_delivery_date_key"),
        col("shipping.date_key").alias("shipping_limit_date_key"),

        col("o.order_status").alias("order_status"),
        col("goi.unit_ordered").alias("unit_ordered"),
        col("goi.total_price").alias("total_price"),
        col("goi.total_freight_value").alias("total_freight_value"),
        col("goi.revenue").alias("total_revenue")
    )
)

In [66]:
# fact_order_sales

In [67]:
# fact_order_sales.select(
#     "purchase_date_key",
#     "approved_date_key",
#     "carrier_date_key",
#     "delivered_date_key",
#     "estimated_delivery_date_key",
#     "shipping_limit_date_key"
# ).show()

In [68]:
import psycopg2

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

with open("truncate_gold.sql", "r", encoding="utf-8") as f:
    sql_script = f.read()

cur.execute(sql_script)

conn.commit()

cur.close()
conn.close()

print("SQL script executed successfully.")

SQL script executed successfully.


In [69]:
tables = [
    ("gold.dim_product", dim_product),
    ("gold.dim_customer", dim_customer),
    ("gold.dim_seller", dim_seller),
    ("gold.dim_date", dim_date),
    ("gold.fact_order_sales", fact_order_sales),
]

for table_name, df in tables:
    try:
        print(f"Writing {table_name}...")

        df.write.jdbc(
            url=jdbc_url,
            table=table_name,
            mode="append",
            properties=connection_properties
        )

        print(f"✓ Successfully wrote {table_name}")

    except Exception as e:
        print(f"✗ Failed to write {table_name}")
        print(e)
        break

Writing gold.dim_product...
✓ Successfully wrote gold.dim_product
Writing gold.dim_customer...
✓ Successfully wrote gold.dim_customer
Writing gold.dim_seller...
✓ Successfully wrote gold.dim_seller
Writing gold.dim_date...
✓ Successfully wrote gold.dim_date
Writing gold.fact_order_sales...
✓ Successfully wrote gold.fact_order_sales
